
# Phase V — Multiscale geometric organization of artistic styles

This phase **does not extract images again** and does not fit another large classification benchmark.

It reuses the 4,000-image ArtBench-10 pilot feature matrix from Phase IV to ask:

\[
\boxed{\text{How is geometric variation organized across style, artist, and spatial scale?}}
\]

Four analyses are produced for both **ArtBench-10** and the **WikiArt-derived 8-style sensitivity subset**:

1. **multiscale geometric fingerprints** of styles;
2. **style-to-style distance matrices and dendrograms** at \(\sigma=\{1,2,4,8\}\);
3. **artist-level permutation tests** for style organization, using only artists with one observed style label so that each artist is an independent unit with a unique class;
4. **nested variance decomposition** \( \text{style}\rightarrow\text{artist}\rightarrow\text{painting} \), again restricted to single-style artists so the hierarchy is statistically well-defined.

The dendrograms are **descriptive geometric similarities**, not art-historical phylogenies, influence graphs, or chronology.


In [ ]:

import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/ardominguezm/painting-geometry.git"
BRANCH = "multiscale-corpus-analysis"
REPO_DIR = Path("/content/painting-geometry")

os.chdir("/content")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
    check=True,
)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Repository:", REPO_DIR)
print("Branch:", BRANCH)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())



## 1. Recover the Phase-IV feature matrix

Upload either:

- `painting_geometry_phase4_artbench_pilot.zip` **recommended**, or
- `artbench_pilot_features.csv`.

The script needs only the already-computed feature table; no ArtBench images are downloaded.


In [ ]:

from google.colab import files
import io, zipfile, pandas as pd, numpy as np

uploaded = files.upload()
INPUT_DIR = Path("/content/phase5_inputs")
INPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = None

for name, data in uploaded.items():
    if name.lower().endswith(".csv") and "artbench_pilot_features" in name:
        FEATURES = INPUT_DIR / "artbench_pilot_features.csv"
        FEATURES.write_bytes(data)
        break

if FEATURES is None:
    for name, data in uploaded.items():
        if name.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(data)) as zf:
                matches = [n for n in zf.namelist() if Path(n).name == "artbench_pilot_features.csv"]
                if matches:
                    FEATURES = INPUT_DIR / "artbench_pilot_features.csv"
                    FEATURES.write_bytes(zf.read(matches[0]))
                    break

if FEATURES is None or not FEATURES.exists():
    raise FileNotFoundError(
        "Upload painting_geometry_phase4_artbench_pilot.zip or artbench_pilot_features.csv."
    )

feat = pd.read_csv(FEATURES)
print("Features:", FEATURES)
print("Shape:", feat.shape)
print("Styles:", feat["style"].nunique())
print("Artists:", feat["artist"].fillna("").astype(str).str.strip().replace("", np.nan).nunique())
print("Curvature features:", sum(c.startswith("geom__curv__") for c in feat.columns))
display(feat.groupby(["split", "style"]).size().unstack(fill_value=0))



## 2. Run Phase V

The default uses 4,999 permutations per scale.

The permutation test first removes artists that carry more than one observed ArtBench style label. This is deliberate: a one-label-per-artist subset makes the artist-level style test and the nested variance decomposition well-defined rather than forcing a false nesting assumption.

The 4,000 images are still used for the descriptive style fingerprints and distance matrices.


In [ ]:

RESULTS = REPO_DIR / "results" / "phase5_style_geometry"
if RESULTS.exists():
    shutil.rmtree(RESULTS)

subprocess.run(
    [
        sys.executable, "-u", "scripts/run_phase5_style_geometry.py",
        "--features", str(FEATURES),
        "--output-dir", str(RESULTS),
        "--n-permutations", "4999",
        "--seed", "42",
    ],
    check=True,
)

summary = pd.read_csv(RESULTS / "phase5_scale_summary.csv")
meta = pd.read_csv(RESULTS / "phase5_metadata.csv")

print("\nMetadata")
display(meta)

print("\nScale summary")
display(summary.round(5))



## 3. Multiscale style fingerprints

Each cell is the median robust \(z\)-score of a curvature descriptor for one style, relative to the global ArtBench distribution.

This is the differential-geometric analogue of a style fingerprint: it shows **which geometric properties are high or low, and at which spatial scale**.


In [ ]:

from IPython.display import Image, display

for dataset in ["artbench10_all", "artbench10_wikiart8"]:
    print("\n", dataset)
    display(Image(filename=str(RESULTS / dataset / "Figure_style_geometric_fingerprint.png")))



## 4. Style distances and geometry-derived dendrograms

For each scale, style centroids are computed in robust-standardized curvature space. Distances are RMS Euclidean distances so that values remain comparable across scales with the same number of curvature summaries.

**Interpretation guardrail:** proximity means similarity in the chosen luminance level-set geometry only. It does not imply historical descent, artistic influence, chronology, or a canonical art-historical taxonomy.


In [ ]:

for dataset in ["artbench10_all", "artbench10_wikiart8"]:
    print("\n====", dataset, "====")
    for sigma in [1, 2, 4, 8]:
        print("sigma =", sigma)
        display(Image(filename=str(RESULTS / dataset / f"Figure_style_distance_sigma{sigma}.png")))
        display(Image(filename=str(RESULTS / dataset / f"Figure_style_dendrogram_sigma{sigma}.png")))



## 5. Does the organization of styles change with scale?

We compare the upper triangles of the style-distance matrices with a Mantel-style Spearman test based on style-label permutations.

High \(\rho\) means the relative geometry of the styles is preserved between two scales; lower \(\rho\) means the style space is reorganized as scale changes.


In [ ]:

for dataset in ["artbench10_all", "artbench10_wikiart8"]:
    corr = pd.read_csv(RESULTS / dataset / "distance_matrix_scale_mantel_correlations.csv")
    print("\n", dataset)
    display(corr.round(5))
    display(Image(filename=str(RESULTS / dataset / "Figure_distance_matrix_scale_correlations.png")))



## 6. Style vs artist vs painting variation

For artists represented by **one observed style label**, each curvature feature is decomposed descriptively as

\[
SS_{\rm total}
=
SS_{\rm style}
+
SS_{\rm artist(style)}
+
SS_{\rm painting}.
\]

This is a nested sum-of-squares decomposition, not a causal model.

We report the median fraction across the ten curvature summaries at each scale. We also perform a multivariate permutation test on **one centroid per single-style artist**, giving every artist equal weight.


In [ ]:

for dataset in ["artbench10_all", "artbench10_wikiart8"]:
    print("\n====", dataset, "====")
    tests = pd.read_csv(RESULTS / dataset / "single_style_artist_centroid_permutation_tests.csv")
    vp = pd.read_csv(RESULTS / dataset / "nested_single_style_variance_partition_features.csv")
    vp_med = (
        vp.groupby("sigma_ref")[[
            "style_fraction",
            "artist_within_style_fraction",
            "painting_residual_fraction",
            "style_share_of_between_artist_variation",
        ]]
        .median()
        .reset_index()
    )
    print("Artist-centroid permutation tests")
    display(tests.round(5))
    print("Median nested variance fractions")
    display(vp_med.round(5))
    display(Image(filename=str(RESULTS / dataset / "Figure_nested_single_style_variance_partition.png")))
    display(Image(filename=str(RESULTS / dataset / "Figure_single_style_artist_centroid_effect.png")))



## 7. Closest and farthest geometric style pairs

These tables help interpret the dendrograms without relying on the visualization alone.


In [ ]:

for dataset in ["artbench10_all", "artbench10_wikiart8"]:
    pairs = pd.read_csv(RESULTS / dataset / "style_pair_distances_by_scale.csv")
    print("\n====", dataset, "====")
    for sigma in [1, 2, 4, 8]:
        p = pairs[pairs["sigma_ref"] == sigma].sort_values("distance")
        print(f"\nsigma={sigma} | nearest")
        display(p[["style_a", "style_b", "distance"]].head(5).round(4))
        print(f"sigma={sigma} | farthest")
        display(p[["style_a", "style_b", "distance"]].tail(5).sort_values("distance", ascending=False).round(4))



## 8. Compact readout

The notebook deliberately does **not** declare a historical hierarchy automatically. The quantities below answer narrower computational questions:

- Which scale maximizes style separation?
- Which scale maximizes the style share in the nested artist hierarchy?
- At which scales is the artist-centroid style effect significant?
- How much does the style-distance geometry reorganize across scales?

These results should be interpreted jointly with Phase IVb classification and Phase III resolution robustness.


In [ ]:

for dataset in ["artbench10_all", "artbench10_wikiart8"]:
    sub = summary[summary["dataset"] == dataset].sort_values("sigma_ref").copy()
    print("\n====", dataset, "====")
    best_sep = sub.loc[sub["mean_pairwise_style_distance"].idxmax()]
    best_style = sub.loc[sub["style_fraction"].idxmax()]
    best_eta = sub.loc[sub["eta2_style_artist_centroids"].idxmax()]
    print("Max mean pairwise style distance: sigma =", int(best_sep["sigma_ref"]),
          "|", round(best_sep["mean_pairwise_style_distance"], 4))
    print("Max median style variance fraction: sigma =", int(best_style["sigma_ref"]),
          "|", round(best_style["style_fraction"], 4))
    print("Max artist-centroid style eta^2: sigma =", int(best_eta["sigma_ref"]),
          "|", round(best_eta["eta2_style_artist_centroids"], 4),
          "| p =", round(best_eta["permutation_p"], 5))

    corr = pd.read_csv(RESULTS / dataset / "distance_matrix_scale_mantel_correlations.csv")
    cross = corr[(corr["sigma_a"] == 1) & (corr["sigma_b"] == 8)]
    if len(cross):
        r = cross.iloc[0]
        print("Distance-space similarity sigma1 vs sigma8:",
              "rho =", round(r["mantel_spearman_rho"], 4),
              "| p =", round(r["permutation_p"], 5))



## 9. Package results

The ZIP contains all CSV tables and publication-oriented figures. Upload it back to ChatGPT for interpretation together with the Phase IVb results.


In [ ]:

import shutil
from google.colab import files

ZIP_BASE = Path("/content/painting_geometry_phase5_style_geometry")
zip_path = shutil.make_archive(str(ZIP_BASE), "zip", root_dir=RESULTS)

print("Created:", zip_path)
files.download(zip_path)
